# Homework 3 — Bayesian Network Inference (Polytree)

**Course:** Artificial Intelligence  
**Names:**  Ali Emre Kaya | 150210097 

**Deliverables:** Jupyter Notebook  
**This version intentionally omits most implementations. Your task is to complete all TODOs.**

**Work individually or in pairs.** Include **full names and student numbers** at the top of this notebook and in your report (part 7).

**Objective:**
Implement and compare:
- Inference by enumeration  
- Inference by variable elimination  
- Belief propagation on a polytree


## 0) Setup


In [1]:
import json
from typing import Dict, List, Tuple


import itertools
from collections import deque


## 1) Load Polytree Network


In [2]:
# bn ; baesian network
with open("network_polytree_hw3.json", "r", encoding="utf-8") as f:
    bn = json.load(f)

variables = bn["variables"]
DOMAIN = bn["domain"]

print("Variables:", [v["name"] for v in variables])
print("Domain:", DOMAIN)

Variables: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U']
Domain: [0, 1]


### 1.1 Data Structures (TODO)

Build parents, children, CPT lookup, and any other structures you need.


In [3]:
# TODO: construct data structures for parents, children, CPT lookup, topo order, etc.

from collections import defaultdict, deque

class Variable:
    def __init__(self, name: str, parents: list[str], cpt_rows: list[dict], domain=None):
        self.name = name
        self.parents = list(parents or [])
        self.children = []  # will be filled by BayesNet
        self.domain = list(domain or [])

        # Build CPT lookup: key = tuple(parent_values in self.parents order)
        # value = [P(self=0), P(self=1)] in domain order
        self.cpt = {}

        for row in (cpt_rows or []):
            when = row.get("when", {}) or {}

            # probablity comes from p_true, always exists
            if "p_true" not in row:
                raise ValueError(f"{name}: CPT row missing 'p_true': {row}")
            
            p1 = float(row["p_true"])
            p0 = 1.0 - p1   # p_false

            key = tuple(int(when[p]) for p in self.parents)  # respects parent order
            self.cpt[key] = [p0, p1]

        print(self.name, " -> ", self.cpt)

    def P(self, value: int, evidence: dict[str, int]) -> float:
        """
        Return P(self=value | parents=evidence[parents]).
        evidence must contain all parent assignments.
        """
        key = tuple(evidence[p] for p in self.parents) if self.parents else ()
        probs = self.cpt[key]  # [P(0), P(1)]
        idx = self.domain.index(value)
        return probs[idx]


class BayesNet:
    def __init__(self, bn_json: dict):
        self.domain = bn_json.get("domain")
        self.vars = {}  # name -> Variable

        # create variable objects
        for v in bn_json["variables"]:
            name = v["name"]
            parents = v.get("parents", []) or []
            cpt_rows = v.get("cpt", []) or []
            self.vars[name] = Variable(name, parents, cpt_rows, domain=self.domain)

        # build children links
        for var in self.vars.values():
            for p in var.parents:
                self.vars[p].children.append(var.name)

        self.names = list(self.vars.keys())

    def topo_order(self) -> list[str]:
        """Kahn topo sort using parents/children."""
        indeg = {n: len(self.vars[n].parents) for n in self.names}
        q = deque([n for n in self.names if indeg[n] == 0])
        order = []
        while q:
            u = q.popleft()
            order.append(u)
            for ch in self.vars[u].children:
                indeg[ch] -= 1
                if indeg[ch] == 0:
                    q.append(ch)
        
        # check
        if len(order) != len(self.names):
            raise ValueError("Cycle detected (not a DAG).")
        return order

    def parents(self, name: str) -> list[str]:
        return self.vars[name].parents

    def children(self, name: str) -> list[str]:
        return self.vars[name].children

    def P(self, var: str, value: int, evidence: dict[str, int]) -> float:
        return self.vars[var].P(value, evidence)


# --- usage ---
net = BayesNet(bn)
print("-"*50)
print(net.topo_order())
print("-"*50)
print(net.parents("B"), net.children("A"))
print("-"*50)
print(net.P("A", 1, {"A": 0}))  # P(B=1 | A=0) should be 0.2 in your example
print("-"*50)


A  ->  {(): [0.4, 0.6]}
B  ->  {(0,): [0.8, 0.2], (1,): [0.19999999999999996, 0.8]}
C  ->  {(0,): [0.7, 0.3], (1,): [0.30000000000000004, 0.7]}
D  ->  {(0,): [0.9, 0.1], (1,): [0.4, 0.6]}
E  ->  {(0,): [0.6, 0.4], (1,): [0.09999999999999998, 0.9]}
F  ->  {(0,): [0.75, 0.25], (1,): [0.25, 0.75]}
G  ->  {(0,): [0.8, 0.2], (1,): [0.15000000000000002, 0.85]}
H  ->  {(0,): [0.95, 0.05], (1,): [0.30000000000000004, 0.7]}
I  ->  {(0,): [0.85, 0.15], (1,): [0.19999999999999996, 0.8]}
J  ->  {(0,): [0.7, 0.3], (1,): [0.09999999999999998, 0.9]}
K  ->  {(0,): [0.8, 0.2], (1,): [0.15000000000000002, 0.85]}
L  ->  {(0,): [0.75, 0.25], (1,): [0.09999999999999998, 0.9]}
M  ->  {(0,): [0.65, 0.35], (1,): [0.19999999999999996, 0.8]}
N  ->  {(0,): [0.9, 0.1], (1,): [0.30000000000000004, 0.7]}
O  ->  {(0,): [0.8, 0.2], (1,): [0.25, 0.75]}
P  ->  {(0,): [0.8, 0.2], (1,): [0.15000000000000002, 0.85]}
Q  ->  {(0,): [0.85, 0.15], (1,): [0.19999999999999996, 0.8]}
R  ->  {(0,): [0.75, 0.25], (1,): [0.25, 0.75

## 2) Inference by Enumeration (TODO)


In [4]:
def enumerate_query(query_var: str, evidence: Dict[str, int], net: BayesNet) -> Dict[int, float]:
    """
    Return P(query_var | evidence) using enumeration.
    """
    vars_order = net.topo_order()
    domain = net.domain

    # sanity check
    for k, v in evidence.items():
        if v not in domain:
            raise ValueError(f"Evidence {k}={v} not in domain {domain}")

    def joint_prob(full_assign: Dict[str, int]) -> float:
        """Compute; P(X | Parents(X)) in topological order."""
        p = 1.0
        for X in vars_order:
            p *= net.P(X, full_assign[X], full_assign)
        return p

    # hidden vars should calculated for all possible domain
    hidden = [X for X in vars_order if X != query_var and X not in evidence]

    unnorm = {}
    for qval in domain:
        base = dict(evidence)
        base[query_var] = qval

        total = 0.0
        for idx ,values in enumerate(itertools.product(domain, repeat=len(hidden))):

            full = dict(base)
            full.update(dict(zip(hidden, values)))
            total += joint_prob(full)

            # little debug
            #if(qval == 0 and idx < 10): print(idx, " -> ", values, " -> ", total)

        unnorm[qval] = total

    # Normalize
    Z = sum(unnorm.values())
    #print("Z :" , Z)
    if Z == 0.0:
        # inconsistent evidence
        return {v: 0.0 for v in domain}

    return {v: unnorm[v] / Z for v in domain}


In [5]:
result = enumerate_query("A", {"B": 1}, net)
print(result)

result = enumerate_query("B", {"A": 1}, net)
print(result)



{0: 0.14285714285713152, 1: 0.8571428571428684}
{0: 0.19999999999997833, 1: 0.8000000000000217}
{0: 0.19999999999997833, 1: 0.8000000000000217}


## 3) Inference by Variable Elimination (TODO)


In [6]:

class Factor:
    """
    Factor over binary variables DOMAIN=[0,1] (prerequires)
      - vars: ordered list of variable names in this factor
      - table: dict[tuple(values aligned with vars)] -> float
    """
    def __init__(self, vars: List[str], table: Dict[Tuple[int, ...], float]):
        self.vars = list(vars)
        self.table = dict(table)

    def copy(self):
        return Factor(self.vars[:], dict(self.table))

    def restrict(self, var: str, value: int) -> "Factor":
        """
        Condition factor on var=value (reduce dimension).
            f(... , var , ...) --> f(... , var=value , ...)
        """
        if var not in self.vars:
            return self.copy()

        i = self.vars.index(var)
        new_vars = self.vars[:i] + self.vars[i+1:]
        new_table = {}
        for key, prob in self.table.items():
            if key[i] == value:
                new_key = key[:i] + key[i+1:]
                new_table[new_key] = prob
        return Factor(new_vars, new_table)

    def sum_out(self, var: str, domain: List[int]) -> "Factor":
        """
        Marginalize out var by summing over its domain.
            f'(A) = f(A,B=0) + f(A,B=1) // Destroy var B
        """
        if var not in self.vars:
            return self.copy()

        i = self.vars.index(var)
        new_vars = self.vars[:i] + self.vars[i+1:]
        new_table = {}

        for assign in itertools.product(domain, repeat=len(new_vars)):
            total = 0.0
            for v in domain:
                full = list(assign)
                full.insert(i, v)
                total += self.table[tuple(full)]
            new_table[tuple(assign)] = total

        return Factor(new_vars, new_table)

    def multiply(self, other: "Factor", domain: List[int]) -> "Factor":
        """Multiply two factors (join on shared vars)."""
        # union of variables, keep stable order: self.vars then others not already included
        new_vars = self.vars[:] + [v for v in other.vars if v not in self.vars]

        new_table = {}
        for assign in itertools.product(domain, repeat=len(new_vars)):
            asg = dict(zip(new_vars, assign))

            key_self = tuple(asg[v] for v in self.vars)
            key_other = tuple(asg[v] for v in other.vars)

            new_table[tuple(assign)] = self.table[key_self] * other.table[key_other]

        return Factor(new_vars, new_table)

    def normalize(self) -> "Factor":
        """Normalize factor to sum to 1"""
        Z = sum(self.table.values())
        if Z == 0.0:
            return self.copy()
        return Factor(self.vars, {k: v / Z for k, v in self.table.items()})

def factor_from_cpt(var_name: str, net: BayesNet) -> Factor:
    """
    With CPT, build factor P(X | Parents(X)).
    CPT -> Factor Table
    """
    var = net.vars[var_name]
    parents = var.parents
    f_vars = parents + [var_name] # ordered

    # iterate over all parent assignments which exist in CPT
    table = {}
    # var.cpt -> parent_tuple -> [P0,P1]
    for pkey, probs01 in var.cpt.items():
        # pkey is tuple aligned with parents
        for xval in net.domain:
            prob = probs01[net.domain.index(xval)]
            key = tuple(pkey) + (xval,)
            table[key] = prob

    return Factor(f_vars, table)


In [7]:

def ve_query(query_var: str, evidence: Dict[str, int], net: BayesNet) -> Dict[int, float]:
    """
    Return P(query_var | evidence) using variable elimination.
    """
    domain = net.domain
    vars_order = net.topo_order()  # for a reasonable elimination order

    # initialize factors with CPT
    factors: List[Factor] = [factor_from_cpt(X, net) for X in vars_order]

    # restrict factors using evidence
    for ev_var, ev_val in evidence.items():
        factors = [f.restrict(ev_var, ev_val) for f in factors]

    # elimination variables: all except query and evidence vars
    elim_vars = [v for v in vars_order if v != query_var and v not in evidence]

    # eliminate
    for z in elim_vars:
        # factors that mention z
        involved = [f for f in factors if z in f.vars]
        if not involved: continue
        rest = [f for f in factors if z not in f.vars]

        # multiply all involved
        prod = involved[0]
        for f in involved[1:]:
            prod = prod.multiply(f, domain)

        # sum out z
        summed = prod.sum_out(z, domain)

        factors = rest + [summed]

    # sanity check
    if not factors:
        raise ValueError("No factors left; check network/evidence.")
    
    # multiply remaining factors
    result_factor = factors[0]
    for f in factors[1:]:
        result_factor = result_factor.multiply(f, domain)

    # now result_factor should be over [query_var]
    for v in list(result_factor.vars):
        if v != query_var:
            result_factor = result_factor.sum_out(v, domain)

    # respectful representation
    result_factor = result_factor.normalize()

    if result_factor.vars != [query_var]:
        raise ValueError(f"Unexpected result vars: {result_factor.vars}")
    
    # convert to {0:p0, 1:p1}
    out = {}
    for qval in domain:
        out[qval] = result_factor.table[(qval,)]
    return out

In [8]:
#print("ENUM:", enumerate_query("B", {"A": 1}, net))
#print("VE  :", ve_query("B", {"A": 1}))

tests = [
    ("A", {}),
    ("A", {"B": 1}),
    ("B", {"A": 0}),
    ("B", {"A": 1}),
]

for q,e in tests:
    pe = enumerate_query(q,e, net)
    pv = ve_query(q,e, net)
    #print(q, e, "ENUM", pe, "VE", pv)


## 4) Belief Propagation on a Polytree (TODO)


In [9]:
def belief_propagation(query_var: str, evidence: Dict[str, int], net: "BayesNet") -> Dict[int, float]:
    """
    Returns: {0: P(query=0|e), 1: P(query=1|e)}
    """

    # basis
    domain = net.domain
    vars_names = list(net.vars.keys())

    # build factor graph
    # one factor per variable # CPT -> f_X(parents..., X) = P(X | parents)
    factor_scope: Dict[str, List[str]] = {}
    factor_table: Dict[str, Dict[Tuple[int, ...], float]] = {}

    for X in vars_names:
        var = net.vars[X]
        fid = f"f_{X}"
        scope = var.parents + [X]
        factor_scope[fid] = scope

        table = {}
        for pkey, probs01 in var.cpt.items():  # pkey aligned with parents
            # probs01 = [P(X=0|p), P(X=1|p)]
            table[tuple(pkey) + (0,)] = probs01[0]
            table[tuple(pkey) + (1,)] = probs01[1]
        factor_table[fid] = table

    # adjacency list
    nbr_v = {v: [] for v in vars_names}   # variable -> factors
    for fid, scope in factor_scope.items():
        for v in scope:
            nbr_v[v].append(fid)

    # evidence potential
    def phi(v: str, x: int) -> float:
        if v in evidence:
            return 1.0 if x == evidence[v] else 0.0
        return 1.0

    # memoized messages on edges of the factor list
    # msg((type,name) -> (type,name), x) where x is 
    #       the value of the receiving variable if receiver is variable,
    # or    the sender variable value if sender is variable
    memo_v_to_f: Dict[Tuple[str, str], List[float]] = {}  # (v, fid) -> [m0,m1]
    memo_f_to_v: Dict[Tuple[str, str], List[float]] = {}  # (fid, v) -> [m0,m1]

    def v_to_f(v: str, fid: str) -> List[float]:
        key = (v, fid)
        if key in memo_v_to_f:
            return memo_v_to_f[key]

        out = []
        for x in domain:
            prod = phi(v, x)
            for f2 in nbr_v[v]:
                if f2 == fid:
                    continue
                prod *= f_to_v(f2, v)[x]  # x is 0/1 so usable as index
            out.append(prod)

        memo_v_to_f[key] = out
        return out

    def f_to_v(fid: str, v: str) -> List[float]:

        key = (fid, v)
        if key in memo_f_to_v:
            return memo_f_to_v[key]

        scope = factor_scope[fid]
        table = factor_table[fid]

        other_vars = [u for u in scope if u != v]
        out = [0.0, 0.0]

        for x_v in domain:
            total = 0.0
            for other_vals in itertools.product(domain, repeat=len(other_vars)):
                assign = {v: x_v}
                assign.update(dict(zip(other_vars, other_vals)))

                fval = table[tuple(assign[u] for u in scope)]
                prod_in = 1.0
                for u in other_vars:
                    prod_in *= v_to_f(u, fid)[assign[u]]  # assign[u] is 0/1
                total += fval * prod_in

            out[x_v] = total

        memo_f_to_v[key] = out
        return out

    # belief at query
    unnorm = {}
    for x in domain:
        b = phi(query_var, x)
        for fid in nbr_v[query_var]:
            b *= f_to_v(fid, query_var)[x]
        unnorm[x] = b

    Z = unnorm[0] + unnorm[1]
    if Z == 0.0:
        return {0: 0.0, 1: 0.0}
    return {0: unnorm[0] / Z, 1: unnorm[1] / Z}


In [10]:
print(belief_propagation("A", {}, net))
print(belief_propagation("B", {"A": 1}, net))
q, e = "B", {"A": 1}
print("BP  :", belief_propagation(q, e, net))
print("ENUM:", enumerate_query(q, e, net))
print("VE  :", ve_query(q, e, net))


{0: 0.4, 1: 0.6}
{0: 0.19999999999999996, 1: 0.8}
BP  : {0: 0.19999999999999996, 1: 0.8}
ENUM: {0: 0.19999999999997833, 1: 0.8000000000000217}
VE  : {0: 0.2, 1: 0.8}
ENUM: {0: 0.19999999999997833, 1: 0.8000000000000217}
VE  : {0: 0.2, 1: 0.8}


## 5) Sanity Checks (TODO)

Pick a few (query, evidence) pairs and compare outputs of all three methods.


In [11]:
bench_queries = [
    ("A", {}),             # prior
    ("B", {"A": 1}),       # simple parent evidence
    ("A", {"H": 1}),                    
    ("C", {"H": 1}),                    
    ("A", {"T": 1}),                   
    ("B", {"T": 1}),                   
    ("A", {"H": 1, "T": 1}), # multi-leaf evidence
    ("B", {"N": 1, "O": 0}),            
]

In [12]:
# Build net (uses the BayesNet/Variable classes you already added)
#net = BayesNet(bn) #defined in first block

print("Domain: ", net.domain)
print("Topo order:", net.topo_order())

# Pick tests automatically from the loaded variables (works for your A..U polytree too)


def pretty(d):
    return {k: round(v, 8) for k, v in d.items()}

def l1_diff(p, q):
    return sum(abs(p[x] - q[x]) for x in p.keys())


for q, e in bench_queries:
    pe = enumerate_query(q, e, net)
    pv = ve_query(q, e, net)
    pb = belief_propagation(q, e, net)

    print(f"\nQuery: P({q} | {e})")
    print("  ENUM:", pretty(pe))
    print("  VE  :", pretty(pv))
    print("  BP  :", pretty(pb))

    if l1_diff(pe, pv) > 1e-6 or l1_diff(pe, pb) > 1e-6:
        print("  !! mismatch > 1e-6 (check implementation / polytree assumption)")

                

Domain:  [0, 1]
Topo order: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U']

Query: P(A | {})
  ENUM: {0: 0.4, 1: 0.6}
  VE  : {0: 0.4, 1: 0.6}
  BP  : {0: 0.4, 1: 0.6}

Query: P(A | {})
  ENUM: {0: 0.4, 1: 0.6}
  VE  : {0: 0.4, 1: 0.6}
  BP  : {0: 0.4, 1: 0.6}

Query: P(B | {'A': 1})
  ENUM: {0: 0.2, 1: 0.8}
  VE  : {0: 0.2, 1: 0.8}
  BP  : {0: 0.2, 1: 0.8}

Query: P(B | {'A': 1})
  ENUM: {0: 0.2, 1: 0.8}
  VE  : {0: 0.2, 1: 0.8}
  BP  : {0: 0.2, 1: 0.8}

Query: P(A | {'H': 1})
  ENUM: {0: 0.24242424, 1: 0.75757576}
  VE  : {0: 0.24242424, 1: 0.75757576}
  BP  : {0: 0.24242424, 1: 0.75757576}

Query: P(A | {'H': 1})
  ENUM: {0: 0.24242424, 1: 0.75757576}
  VE  : {0: 0.24242424, 1: 0.75757576}
  BP  : {0: 0.24242424, 1: 0.75757576}

Query: P(C | {'H': 1})
  ENUM: {0: 0.3969697, 1: 0.6030303}
  VE  : {0: 0.3969697, 1: 0.6030303}
  BP  : {0: 0.3969697, 1: 0.6030303}

Query: P(C | {'H': 1})
  ENUM: {0: 0.3969697, 1: 0.6030303}
  VE

## 6) Timing Experiments (TODO)


In [13]:
def time_method(method, query_var: str, evidence: Dict[str, int], net: BayesNet, repeat: int = 1) -> float:
    total = 0.0
    for _ in range(repeat):
        start = time.time()
        _ = method(query_var, evidence, net)
        end = time.time()
        total += (end - start)
    return total / repeat

# TODO: define queries and compare runtimes across methods

import time

def ms(x): 
    return round(1000*x, 3)

print(f"{'Query':<22} {'ENUM (ms)':>10} {'VE (ms)':>10} {'BP (ms)':>10}")
print("-"*56)

for q, e in bench_queries:

    enumerate_query(q, e, net)
    ve_query(q, e, net)
    belief_propagation(q, e, net)

    t_enum = time_method(enumerate_query, q, e, net)
    t_ve   = time_method(ve_query, q, e, net)
    t_bp   = time_method(belief_propagation, q, e, net)

    label = f"P({q}|{e})"
    print(f"{label:<22} {ms(t_enum):>10} {ms(t_ve):>10} {ms(t_bp):>10}")


Query                   ENUM (ms)    VE (ms)    BP (ms)
--------------------------------------------------------
P(A|{})                 23234.905      0.607      0.125
P(A|{})                 23234.905      0.607      0.125
P(B|{'A': 1})           11640.817      0.669      0.119
P(B|{'A': 1})           11640.817      0.669      0.119
P(A|{'H': 1})           12088.319      0.845      0.128
P(A|{'H': 1})           12088.319      0.845      0.128
P(C|{'H': 1})           11961.578      0.902      0.117
P(C|{'H': 1})           11961.578      0.902      0.117
P(A|{'T': 1})           11771.194      0.707      0.171
P(A|{'T': 1})           11771.194      0.707      0.171
P(B|{'T': 1})           11575.947      0.553      0.223
P(B|{'T': 1})           11575.947      0.553      0.223
P(A|{'H': 1, 'T': 1})    6204.612      0.622      0.159
P(A|{'H': 1, 'T': 1})    6204.612      0.622      0.159
P(B|{'N': 1, 'O': 0})    5717.445       0.54       0.15
P(B|{'N': 1, 'O': 0})    5717.445       0.54   

## 7) Short Report

### 0. Author
Ali Emre Kaya, 150210097

### 1. Introduction

This homework implements and compares three fundamental inference algorithms for Bayesian Networks:
1. **Inference by Enumeration**
2. **Variable Elimination (VE)**
3. **Belief Propagation (BP)**

The experiments completed with using network_polytree_hw3.json file. This file has 21 nodes A to U.

---

### 2. Design Choices

#### 2.1 Data Structures
- **Variable class**: Encapsulates each node's name, parents, children, domain, and CPT. The CPT is a dictionary and map parent assignment tuples to probability distributions like [P(X=0), P(X=1)].
- **BayesNet class**: Manages the full network, providing topological ordering and probability lookups.
- **Factor class**: Used in VE and BP; stores a scope (variable list) and a table mapping value tuples to probabilities.

#### 2.2 Enumeration
- Computes P(Q | E) by iterating over all assignments to hidden variables (variables which are not mentioned in Q or E) in topological order.
- Straightforward but exponential in the number of hidden variables: O(d^n); d -> domain size, n -> hidden node count. In our case 2^(20-e) which is too slow.

#### 2.3 Variable Elimination
- Start with probability tables (CPTs).
- Apply the evidence to reduce tables.
- Remove hidden variables: Multiply related tables, Sum out the variable.
- Reuse intermediate results for simplify.

#### 2.4 Belief Propagation
- The network is turned into a graph of variables and factors.
- Nodes send messages to each other.
- Messages pass probabilities back and forth -> results are stored so they are not recomputed.
- In a polytree, it gives exact results.

---

### 3. Consistency of Results

All three methods produce **identical posterior distributions** (within floating-point tolerance of $10^{-6}$) across all test queries:

| Query | Evidence | ENUM | VE | BP |
|-------|----------|------|----|----|
| P(A) | {} | {0: 0.4, 1: 0.6} | same | same |
| P(B) | {A=1} | {0: 0.2, 1: 0.8} | same | same |
| P(A) | {H=1} | {0: 0.24242424, 1: 0.75757576} | same | same |
| P(A) | {H=1, T=1} | {0: 0.21476046, 1: 0.78523954} | same | same |

This confirms correctness of all implementations and validates that the network satisfies the polytree assumption (no undirected cycles), making BP exact.

---

### 4. Timing Behavior

| Query | ENUM (ms) | VE (ms) | BP (ms) |
|-------|-----------|---------|---------|
| P(A \| {}) | ~23319 | ~1 | ~0 |
| P(B \| {A=1}) | ~11073 | ~0 | ~0 |
| P(A \| {H=1}) | ~11547 | ~1 | ~0 |
| P(A \| {H=1, T=1}) | ~5678 | ~1| ~0 |

**Observations:**
1. **Enumeration** is consistently the slowest due to its exponential complexity O(2^{19}) for 21 binary nodes (minus query and evidence(s)).
2. **Variable Elimination** offers significant speedup by avoiding repeated computations through factor multiplication and marginalization.
3. **Belief Propagation** is the fastest for this polytree structure, benefiting from:
   - Linear message passing
   - Memoization preventing redundant recursive calls
   - Natural fit for tree-structured graphical models

---

### 5. Complexity Comparison

| Method | Time Complexity | Space Complexity | Exactness |
|--------|-----------------|------------------|-----------|
| Enumeration | $O(d^n)$ | $O(n)$ | Exact |
| Variable Elimination | $O(d^{w+1})$ where $w$ = treewidth | $O(d^w)$ | Exact |
| Belief Propagation | $O(n \cdot d^k)$ where $k$ = max factor size | $O(n \cdot d)$ | Exact on trees/polytrees |


---

### 6. Conclusion

With this homework I implement and compare three inference method:
- **Enumeration** is simple but not practical for large networks
- **Variable Elimination** provides a good balance of generality and efficiency
- **Belief Propagation** excellent on tree-structured networks with guaranteed exactness and optimal runtime (linear)

The implementations successfully replicate theoretical expectations: all methods agree on posteriors, and BP outperforms alternatives on the given polytree structure.